[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/099.%20The%20Electric%20Vehicle%20Routing%20Problem%20%28EVRP%29/P99-Tier-5_executed.ipynb)

---


# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 5: Integrated Digital Twin — Fleet-Wide Energy Simulation

### Purpose and outputs

This notebook builds a small, simulated digital twin for an electric-vehicle fleet. It creates virtual representations of vehicles, charging stations, and their environment; generates synthetic sensor readings; synchronizes the virtual assets with those readings; estimates energy-related indicators; and explores illustrative what-if scenarios. The outputs include simulated sensor data, key performance indicators (KPIs), predictive heuristics, a dashboard, and a performance analysis.

The workflow is a simulation and monitoring exercise, not a route-optimization model: it does not select routes or solve an EVRP optimization problem. Its solution method is the step-by-step digital-twin workflow demonstrated here: define the virtual assets, generate data, synchronize the twins, calculate indicators, and examine predictions and scenarios.

### Model assumptions and scope

- Vehicle locations, station locations, and route coordinates use a two-dimensional plane; distances between route points are Euclidean and are interpreted as kilometres.
- Battery level and battery capacity are measured in kWh. Battery health and utilization are fractions between 0 and 1; battery percentage is derived from level divided by capacity.
- Sensor readings are synthetic and include random measurement noise. They are not observations from physical vehicles or infrastructure.
- The energy-consumption and charging relationships are illustrative approximations. No vehicle movement, customer deliveries, charging-session assignment, or physical-world synchronization occurs.
- Demand and charging-time predictions are rule-based heuristics, not fitted forecasting models. The service-level KPI is an assumed illustrative value, not a measured delivery outcome.
- What-if scenarios temporarily modify the simulated system, calculate scenario results, and then restore the baseline state. Their results are not evidence of real-world performance.

### Workflow

1. Represent vehicles, charging stations, environmental conditions, and the digital-twin system as data structures.
2. Generate timestamped synthetic IoT readings with sensor-quality scores.
3. Estimate route energy use and charging outcomes using the stated illustrative relationships.
4. Produce simple charging-demand, battery-degradation, and charging-time predictions.
5. Orchestrate synchronization, KPI calculations, and isolated what-if scenarios.
6. Visualize and interpret the simulated state and its limitations.

### Example fleet

The example contains one depot, four electric vehicles, and three charging stations. Each vehicle has a nominal battery capacity of 100 kWh. The example provides synthetic sensor readings for vehicle, station, and environmental variables. The code determines the number of generated readings from the configured sensors and simulation duration; it does not claim to represent a physical deployment.

In [ ]:
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Any, Dict, List, Tuple
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the synthetic sensor-noise sequence reproducible.
np.random.seed(42)
random.seed(42)

## 1. Represent the digital twin

A digital twin is a virtual representation of assets and the information associated with them. Here, dataclasses make the structure of each asset explicit: a vehicle has a location, battery state, operating conditions, and sensor values; a charging station has capacity, availability, and energy attributes; and the system groups these assets with environmental data and histories.

The distinction between battery energy and battery percentage matters. `battery_level` and `battery_capacity` are both in kWh; a percentage is calculated as `100 * battery_level / battery_capacity`. This keeps the units consistent when estimating energy use and displaying status.

In [ ]:
@dataclass
class IoTDataPoint:
    """One timestamped synthetic sensor reading."""

    timestamp: datetime
    sensor_id: str
    value: float
    unit: str
    quality_score: float


@dataclass
class VehicleTwin:
    """Virtual representation of one electric vehicle."""

    vehicle_id: int
    location: Tuple[float, float]
    battery_level: float  # kWh
    battery_capacity: float  # kWh
    battery_health: float  # Fraction from 0 to 1
    speed: float  # km/h
    payload: float  # kg
    temperature: float  # °C
    energy_consumption_rate: float  # kWh/km
    route: List[int] = field(default_factory=list)
    sensors: Dict[str, Any] = field(default_factory=dict)
    last_update: datetime = field(default_factory=datetime.now)


@dataclass
class ChargingStationTwin:
    """Virtual representation of one charging station."""

    station_id: int
    location: Tuple[float, float]
    capacity: int
    available_points: int
    charging_rate: float  # kW per charging point
    queue_length: int
    utilization_rate: float  # Fraction from 0 to 1
    energy_price: float  # Currency units per kWh
    grid_carbon_intensity: float  # gCO2/kWh
    sensors: Dict[str, Any] = field(default_factory=dict)
    last_update: datetime = field(default_factory=datetime.now)


@dataclass
class EnvironmentTwin:
    """Virtual representation of environmental conditions."""

    temperature: float  # °C
    humidity: float  # Percent
    wind_speed: float  # km/h
    traffic_density: Dict[str, float]
    road_conditions: Dict[str, str]
    carbon_intensity: float  # gCO2/kWh


@dataclass
class DigitalTwinSystem:
    """Fleet assets, sensor stream, and recorded KPI histories."""

    vehicles: List[VehicleTwin]
    charging_stations: List[ChargingStationTwin]
    environment: EnvironmentTwin
    iot_data_stream: deque
    synchronization_interval: float  # seconds
    prediction_horizon: timedelta
    kpi_history: Dict[str, List[float]]


def create_digital_twin_instance() -> DigitalTwinSystem:
    """Build the example fleet and its synthetic operating context."""
    vehicle_specs = [
        (0, 95.0, 0.98, 500.0, 22.0, 0.80, 0.95, 0.99, 0.92, 32.5),
        (1, 88.0, 0.96, 450.0, 21.8, 0.78, 0.94, 0.98, 0.90, 31.8),
        (2, 92.0, 0.99, 520.0, 22.3, 0.82, 0.96, 0.99, 0.93, 33.1),
        (3, 85.0, 0.97, 480.0, 21.9, 0.79, 0.95, 0.98, 0.91, 32.2),
    ]
    vehicles = []
    for (vehicle_id, battery_level, health, payload, temperature, rate,
         gps_accuracy, battery_accuracy, motor_efficiency, tire_pressure) in vehicle_specs:
        vehicles.append(
            VehicleTwin(
                vehicle_id=vehicle_id,
                location=(0.0, 0.0),
                battery_level=battery_level,
                battery_capacity=100.0,
                battery_health=health,
                speed=0.0,
                payload=payload,
                temperature=temperature,
                energy_consumption_rate=rate,
                sensors={
                    "gps_accuracy": gps_accuracy,
                    "battery_sensor_accuracy": battery_accuracy,
                    "temperature_sensor": temperature,
                    "motor_efficiency": motor_efficiency,
                    "tire_pressure": tire_pressure,
                },
            )
        )

    station_specs = [
        (0, (8.0, 12.0), 4, 50.0, 0.25, 150.0, 0.94, 49.8, 25.3),
        (1, (16.0, 8.0), 3, 75.0, 0.28, 165.0, 0.96, 74.5, 26.1),
        (2, (12.0, 18.0), 2, 100.0, 0.32, 180.0, 0.97, 99.2, 27.2),
    ]
    charging_stations = []
    for (station_id, location, capacity, rate, price, carbon,
         efficiency, power_output, temperature) in station_specs:
        charging_stations.append(
            ChargingStationTwin(
                station_id=station_id,
                location=location,
                capacity=capacity,
                available_points=capacity,
                charging_rate=rate,
                queue_length=0,
                utilization_rate=0.0,
                energy_price=price,
                grid_carbon_intensity=carbon,
                sensors={
                    "power_output": power_output,
                    "efficiency": efficiency,
                    "availability": 1.0,
                    "connector_status": [1] * capacity,
                    "temperature": temperature,
                },
            )
        )

    environment = EnvironmentTwin(
        temperature=22.0,
        humidity=65.0,
        wind_speed=12.0,
        traffic_density={
            "downtown": 0.8,
            "suburban": 0.4,
            "industrial": 0.6,
            "highway": 0.3,
        },
        road_conditions={
            "downtown": "moderate",
            "suburban": "good",
            "industrial": "poor",
            "highway": "good",
        },
        carbon_intensity=160.0,
    )

    return DigitalTwinSystem(
        vehicles=vehicles,
        charging_stations=charging_stations,
        environment=environment,
        iot_data_stream=deque(maxlen=10000),
        synchronization_interval=1.0,
        prediction_horizon=timedelta(hours=2),
        kpi_history={
            "total_energy_consumed": [],
            "average_battery_level": [],
            "fleet_utilization": [],
            "charging_station_utilization": [],
            "carbon_emissions": [],
            "operational_costs": [],
            "service_level": [],
            "prediction_accuracy": [],
        },
    )


digital_twin = create_digital_twin_instance()
print("Created the example digital twin:")
print(f"  Vehicle twins: {len(digital_twin.vehicles)}")
print(f"  Charging-station twins: {len(digital_twin.charging_stations)}")
print(f"  Synchronization interval: {digital_twin.synchronization_interval:.1f} second(s)")
print(f"  Prediction horizon: {digital_twin.prediction_horizon}")

## 2. Generate synthetic IoT data

A sensor stream provides timestamped readings that can be matched to the virtual assets. Each `IoTDataPoint` contains a sensor identifier, a value, a unit, a timestamp, and a quality score. The simulator uses the fixed random seed established above, so the random noise sequence can be reproduced when the notebook is run from the beginning.

For a sensor, the simulator usually adds modest Gaussian noise and occasionally adds larger noise to represent a lower-quality reading. These generated values are demonstration data; they should not be interpreted as measurements or as a calibrated model of sensor failure.

In [ ]:
class IoTDataSimulator:
    """Generate noisy synthetic readings for fleet assets."""

    def __init__(self, digital_twin: DigitalTwinSystem):
        self.digital_twin = digital_twin
        self.sensor_error_rate = 0.02
        self.data_latency = 0.1  # Illustrative average latency in seconds.

    def _noise(self, typical_scale: float, degraded_scale: float) -> float:
        """Sample a reading error, allowing occasional larger deviations."""
        scale = degraded_scale if np.random.random() < self.sensor_error_rate else typical_scale
        return float(np.random.normal(0.0, scale))

    def generate_vehicle_sensor_data(
        self, vehicle: VehicleTwin, timestamp: datetime
    ) -> List[IoTDataPoint]:
        """Generate five readings for one vehicle at one timestamp."""
        readings = [
            ("gps", self._noise(0.5, 5.0), "m", 0.95, 0.70),
            (
                "battery",
                vehicle.battery_level + self._noise(0.5, 2.0),
                "kWh",
                0.98,
                0.80,
            ),
            (
                "speed",
                max(0.0, vehicle.speed + self._noise(1.0, 5.0)),
                "km/h",
                0.96,
                0.75,
            ),
            (
                "consumption",
                max(
                    0.0,
                    vehicle.energy_consumption_rate + self._noise(0.02, 0.1),
                ),
                "kWh/km",
                0.94,
                0.70,
            ),
            (
                "temperature",
                vehicle.temperature + self._noise(0.5, 2.0),
                "°C",
                0.97,
                0.80,
            ),
        ]
        return [
            IoTDataPoint(
                timestamp=timestamp,
                sensor_id=f"vehicle_{vehicle.vehicle_id}_{name}",
                value=float(value),
                unit=unit,
                quality_score=good_quality if np.random.random() >= self.sensor_error_rate else low_quality,
            )
            for name, value, unit, good_quality, low_quality in readings
        ]

    def generate_charging_station_data(
        self, station: ChargingStationTwin, timestamp: datetime
    ) -> List[IoTDataPoint]:
        """Generate power, availability, and queue readings for a station."""
        actual_power = station.charging_rate * (station.capacity - station.available_points)
        power = max(0.0, actual_power + self._noise(1.0, 5.0))
        readings = [
            ("power", power, "kW", 0.96, 0.80),
            ("availability", station.available_points, "points", 0.99, 0.90),
            ("queue", station.queue_length, "vehicles", 0.98, 0.85),
        ]
        return [
            IoTDataPoint(
                timestamp=timestamp,
                sensor_id=f"station_{station.station_id}_{name}",
                value=float(value),
                unit=unit,
                quality_score=good_quality if np.random.random() >= self.sensor_error_rate else low_quality,
            )
            for name, value, unit, good_quality, low_quality in readings
        ]

    def generate_environmental_data(self, timestamp: datetime) -> List[IoTDataPoint]:
        """Generate synthetic temperature and grid-carbon readings."""
        environment = self.digital_twin.environment
        readings = [
            IoTDataPoint(
                timestamp=timestamp,
                sensor_id="env_temperature",
                value=float(environment.temperature + np.random.normal(0.0, 0.2)),
                unit="°C",
                quality_score=0.97,
            ),
            IoTDataPoint(
                timestamp=timestamp,
                sensor_id="grid_carbon",
                value=float(max(0.0, environment.carbon_intensity + np.random.normal(0.0, 5.0))),
                unit="gCO2/kWh",
                quality_score=0.95,
            ),
        ]
        return readings

    def simulate_iot_stream(
        self, duration_seconds: int = 60, start_time: datetime | None = None
    ) -> int:
        """Append readings at one-second intervals and return their count."""
        if duration_seconds < 0:
            raise ValueError("duration_seconds must be non-negative")
        start_time = start_time or datetime.now()
        generated = 0
        for second in range(duration_seconds):
            timestamp = start_time + timedelta(seconds=second)
            readings = []
            for vehicle in self.digital_twin.vehicles:
                readings.extend(self.generate_vehicle_sensor_data(vehicle, timestamp))
            for station in self.digital_twin.charging_stations:
                readings.extend(self.generate_charging_station_data(station, timestamp))
            readings.extend(self.generate_environmental_data(timestamp))
            self.digital_twin.iot_data_stream.extend(readings)
            generated += len(readings)
        return generated


iot_simulator = IoTDataSimulator(digital_twin)
print(f"Created the synthetic IoT simulator with a {iot_simulator.sensor_error_rate:.1%} higher-noise probability.")

## 3. Estimate energy use and charging outcomes

Energy use over a route depends on distance and adjustment factors for traffic, temperature, payload, battery health, and speed. The model multiplies the base route energy by these factors:

$$
E = r d \cdot f_{\mathrm{traffic}} \cdot f_{\mathrm{temperature}} \cdot f_{\mathrm{payload}} \cdot f_{\mathrm{health}} \cdot f_{\mathrm{speed}},
$$

where $r$ is the vehicle's base consumption rate in kWh/km, $d$ is distance in km, and $E$ is estimated energy in kWh. The factors are illustrative approximations defined in the code; they are not empirically validated physical laws.

Charging rate is measured in kW, while charging time is supplied in seconds. Since one kW is one kWh per hour, the energy added is based on rate multiplied by time divided by 3,600, and it cannot exceed the remaining battery capacity. Route distances are calculated from two-dimensional coordinates with the Euclidean formula, so the coordinates are interpreted as kilometres.

In [ ]:
class EnergySimulationEngine:
    """Estimate route energy, charging outcomes, and route energy needs."""

    def __init__(self, digital_twin: DigitalTwinSystem):
        self.digital_twin = digital_twin

    def calculate_energy_consumption(
        self,
        vehicle: VehicleTwin,
        distance: float,
        traffic_factor: float,
        temperature_factor: float,
    ) -> float:
        """Estimate route energy use from base rate and adjustment factors."""
        if distance < 0:
            raise ValueError("distance must be non-negative")
        base_consumption = vehicle.energy_consumption_rate * distance
        traffic_impact = 1.0 + (traffic_factor - 0.5) * 0.3
        payload_impact = 1.0 + (vehicle.payload / 1000.0) * 0.1
        health_impact = 1.0 + (1.0 - vehicle.battery_health) * 0.2
        speed_impact = 1.0 + (abs(vehicle.speed - 55.0) / 50.0) * 0.1
        return float(
            base_consumption
            * traffic_impact
            * temperature_factor
            * payload_impact
            * health_impact
            * speed_impact
        )

    def simulate_charging_behavior(
        self,
        vehicle: VehicleTwin,
        station: ChargingStationTwin,
        charging_time: float,
    ) -> Dict[str, float]:
        """Estimate energy, cost, and emissions for a charging interval in seconds."""
        if charging_time < 0:
            raise ValueError("charging_time must be non-negative")
        battery_level_factor = 1.0 - (vehicle.battery_level / vehicle.battery_capacity) * 0.3
        temperature_factor = max(0.0, 1.0 - abs(vehicle.temperature - 25.0) / 50.0)
        efficiency = float(station.sensors.get("efficiency", 0.95))
        actual_rate = station.charging_rate * battery_level_factor * temperature_factor * efficiency
        energy_added = min(
            actual_rate * charging_time / 3600.0,
            max(0.0, vehicle.battery_capacity - vehicle.battery_level),
        )
        cost = energy_added * station.energy_price
        carbon_emissions = energy_added * station.grid_carbon_intensity / 1000.0
        return {
            "energy_added": energy_added,
            "actual_rate": actual_rate,
            "cost": cost,
            "carbon_emissions": carbon_emissions,
            "final_battery_level": vehicle.battery_level + energy_added,
        }

    def predict_energy_needs(
        self,
        vehicle: VehicleTwin,
        planned_route: List[Tuple[float, float]],
        time_horizon: timedelta,
    ) -> Dict[str, Any]:
        """Estimate route distance, energy needs, and a charging-stop heuristic."""
        if time_horizon.total_seconds() < 0:
            raise ValueError("time_horizon must be non-negative")
        total_distance = sum(
            float(np.hypot(
                planned_route[index + 1][0] - planned_route[index][0],
                planned_route[index + 1][1] - planned_route[index][1],
            ))
            for index in range(len(planned_route) - 1)
        )
        traffic_values = list(self.digital_twin.environment.traffic_density.values())
        avg_traffic_factor = float(np.mean(traffic_values)) if traffic_values else 0.5
        temperature_factor = 1.0 + abs(vehicle.temperature - 20.0) / 20.0 * 0.15
        estimated_consumption = self.calculate_energy_consumption(
            vehicle, total_distance, avg_traffic_factor, temperature_factor
        )
        charging_stops_needed = 0
        if estimated_consumption > vehicle.battery_level * 0.8:
            usable_energy_per_stop = vehicle.battery_capacity * 0.6
            charging_stops_needed = max(
                1, int(np.ceil(estimated_consumption / usable_energy_per_stop))
            )
        return {
            "total_distance": total_distance,
            "estimated_energy": estimated_consumption,
            "charging_stops_needed": charging_stops_needed,
            "can_complete_route": estimated_consumption <= vehicle.battery_level,
            "recommended_charging": max(
                0.0, estimated_consumption - vehicle.battery_level * 0.8
            ),
        }


energy_simulator = EnergySimulationEngine(digital_twin)
print("Created the energy simulation engine.")

## 4. Produce rule-based predictions

The prediction component applies transparent heuristics to the current simulated state. Charging-demand estimates scale current station utilization according to broad time-of-day bands. Battery-degradation estimates extrapolate a small daily rate, adjusted for speed and temperature. Charging-time rankings combine illustrative price and carbon multipliers.

These calculations are useful for demonstrating how a digital twin can organize scenario inputs and outputs. They do not learn from historical data, quantify forecast uncertainty, or establish that a recommended hour is operationally feasible. The requested prediction horizon is reported as context; the demand heuristic itself does not vary with horizon length.

In [ ]:
class PredictiveAnalytics:
    """Provide simple, explicitly heuristic fleet predictions."""

    def __init__(self, digital_twin: DigitalTwinSystem):
        self.digital_twin = digital_twin
        self.prediction_models = {}

    def predict_charging_demand(self, time_horizon: timedelta) -> Dict[str, float]:
        """Scale current station utilization using time-of-day rules."""
        if time_horizon.total_seconds() < 0:
            raise ValueError("time_horizon must be non-negative")
        current_hour = datetime.now().hour
        if 8 <= current_hour <= 18:
            time_multiplier = 1.5
        elif 18 <= current_hour <= 22:
            time_multiplier = 1.2
        else:
            time_multiplier = 0.6
        return {
            f"station_{station.station_id}": min(
                1.0, station.utilization_rate * time_multiplier
            )
            for station in self.digital_twin.charging_stations
        }

    def predict_battery_degradation(
        self, vehicle: VehicleTwin, days_ahead: int = 30
    ) -> Dict[str, float]:
        """Extrapolate a rule-based battery-health decline."""
        if days_ahead < 0:
            raise ValueError("days_ahead must be non-negative")
        daily_degradation_rate = 0.0001
        usage_factor = 1.0 + (vehicle.speed / 50.0) * 0.2
        temperature_factor = 1.0 + abs(vehicle.temperature - 20.0) / 30.0 * 0.3
        adjusted_rate = daily_degradation_rate * usage_factor * temperature_factor
        predicted_health = max(0.7, vehicle.battery_health - adjusted_rate * days_ahead)
        if adjusted_rate > 0:
            days_to_80_percent = max(
                0, int((vehicle.battery_health - 0.8) / adjusted_rate)
            )
        else:
            days_to_80_percent = 0
        return {
            "current_health": vehicle.battery_health,
            "predicted_health": predicted_health,
            "degradation_rate": adjusted_rate,
            "days_to_80_percent": days_to_80_percent,
        }

    def predict_optimal_charging_times(self, vehicle: VehicleTwin) -> Dict[str, Any]:
        """Rank the next 24 hours with illustrative price and carbon scores."""
        if not self.digital_twin.charging_stations:
            raise ValueError("At least one charging station is required")
        current_hour = datetime.now().hour
        avg_price = float(np.mean([
            station.energy_price for station in self.digital_twin.charging_stations
        ]))
        avg_carbon = float(np.mean([
            station.grid_carbon_intensity
            for station in self.digital_twin.charging_stations
        ]))
        ranked_times = []
        for offset in range(24):
            check_hour = (current_hour + offset) % 24
            if 8 <= check_hour <= 18:
                price_multiplier = 1.3
            elif 18 <= check_hour <= 22:
                price_multiplier = 1.1
            else:
                price_multiplier = 0.8
            if 10 <= check_hour <= 16:
                carbon_multiplier = 0.7
            elif 0 <= check_hour <= 6:
                carbon_multiplier = 0.8
            else:
                carbon_multiplier = 1.2
            score = (avg_price * price_multiplier) * 0.6 + (
                avg_carbon * carbon_multiplier
            ) * 0.4
            ranked_times.append({
                "hour": check_hour,
                "price_multiplier": price_multiplier,
                "carbon_multiplier": carbon_multiplier,
                "total_score": score,
                "rank": 0,
            })
        ranked_times.sort(key=lambda item: item["total_score"])
        for rank, time_info in enumerate(ranked_times, start=1):
            time_info["rank"] = rank
        return {
            "optimal_times": ranked_times[:6],
            "best_hour": ranked_times[0]["hour"],
            "worst_hour": ranked_times[-1]["hour"],
        }


predictive_analytics = PredictiveAnalytics(digital_twin)
print("Created the rule-based predictive analytics module.")

## 5. Orchestrate synchronization and KPI calculations

The orchestrator coordinates the simulator, energy estimates, predictions, and KPI history. At each simulated second, it uses recent readings to update the virtual asset state. Sensor timestamps are aligned with the simulation clock so that synchronization uses readings from the intended time window.

The recorded KPI called `total_energy_consumed` is defined here as the fleet's **energy deficit relative to every vehicle being fully charged**: the sum of capacity minus current battery level. It is a state indicator, not cumulative energy consumed during the simulation. Carbon emissions are an illustrative estimate associated with that deficit and the environment's grid intensity. Operational cost is a short-window estimate based on currently occupied charging points. The service-level value is explicitly assumed rather than inferred from deliveries.

In [ ]:
class DigitalTwinOrchestrator:
    """Coordinate sensor generation, synchronization, and KPI recording."""

    def __init__(self, digital_twin: DigitalTwinSystem):
        self.digital_twin = digital_twin
        self.iot_simulator = IoTDataSimulator(digital_twin)
        self.energy_simulator = EnergySimulationEngine(digital_twin)
        self.predictive_analytics = PredictiveAnalytics(digital_twin)
        self.current_time = datetime.now()
        self.simulation_step = 0

    def synchronize_physical_to_virtual(self, timestamp: datetime) -> None:
        """Apply recent synthetic readings to their corresponding virtual assets."""
        recent_data = []
        for data_point in reversed(self.digital_twin.iot_data_stream):
            age = (timestamp - data_point.timestamp).total_seconds()
            if age > 2.0:
                break
            if 0.0 <= age <= 2.0:
                recent_data.append(data_point)

        for vehicle in self.digital_twin.vehicles:
            prefix = f"vehicle_{vehicle.vehicle_id}_"
            for data_point in recent_data:
                if not data_point.sensor_id.startswith(prefix):
                    continue
                if data_point.sensor_id.endswith("_battery"):
                    vehicle.battery_level = min(
                        vehicle.battery_capacity,
                        max(0.0, data_point.value),
                    )
                elif data_point.sensor_id.endswith("_speed"):
                    vehicle.speed = max(0.0, data_point.value)
                elif data_point.sensor_id.endswith("_consumption"):
                    vehicle.energy_consumption_rate = max(0.0, data_point.value)
                elif data_point.sensor_id.endswith("_temperature"):
                    vehicle.temperature = data_point.value
            vehicle.last_update = timestamp

        for station in self.digital_twin.charging_stations:
            prefix = f"station_{station.station_id}_"
            for data_point in recent_data:
                if not data_point.sensor_id.startswith(prefix):
                    continue
                if data_point.sensor_id.endswith("_availability"):
                    station.available_points = min(
                        station.capacity,
                        max(0, int(round(data_point.value))),
                    )
                elif data_point.sensor_id.endswith("_queue"):
                    station.queue_length = max(0, int(round(data_point.value)))
                elif data_point.sensor_id.endswith("_power"):
                    max_power = station.charging_rate * station.capacity
                    station.utilization_rate = (
                        min(1.0, max(0.0, data_point.value / max_power))
                        if max_power > 0
                        else 0.0
                    )
            station.last_update = timestamp

    def calculate_kpis(self, timestamp: datetime) -> Dict[str, float]:
        """Calculate state indicators and explicitly labeled estimates."""
        del timestamp  # The argument keeps the KPI interface tied to simulation time.
        vehicles = self.digital_twin.vehicles
        stations = self.digital_twin.charging_stations
        if not vehicles or not stations:
            raise ValueError("KPI calculation requires vehicles and charging stations")

        energy_deficit = sum(
            max(0.0, vehicle.battery_capacity - vehicle.battery_level)
            for vehicle in vehicles
        )
        average_battery_percent = float(np.mean([
            100.0 * vehicle.battery_level / vehicle.battery_capacity
            for vehicle in vehicles
        ]))
        fleet_utilization = sum(vehicle.speed > 0 for vehicle in vehicles) / len(vehicles)
        average_station_utilization = float(np.mean([
            station.utilization_rate for station in stations
        ]))
        carbon_estimate = (
            energy_deficit * self.digital_twin.environment.carbon_intensity / 1000.0
        )
        estimated_cost = sum(
            station.charging_rate
            * (station.capacity - station.available_points)
            * (0.1 / 60.0)
            * station.energy_price
            for station in stations
        )
        return {
            "total_energy_consumed": energy_deficit,
            "average_battery_level": average_battery_percent,
            "fleet_utilization": fleet_utilization,
            "charging_station_utilization": average_station_utilization,
            "carbon_emissions": carbon_estimate,
            "operational_costs": estimated_cost,
            "service_level": 0.95,
        }

    def run_simulation_step(self, duration_minutes: int = 60) -> Dict[str, Any]:
        """Run a step lasting the specified number of simulated minutes."""
        if duration_minutes < 0:
            raise ValueError("duration_minutes must be non-negative")
        step_start = time.perf_counter()
        duration_seconds = duration_minutes * 60
        start_time = self.current_time
        data_points = self.iot_simulator.simulate_iot_stream(
            duration_seconds=duration_seconds,
            start_time=start_time,
        )
        for second in range(duration_seconds):
            timestamp = start_time + timedelta(seconds=second)
            self.synchronize_physical_to_virtual(timestamp)
            if second % 10 == 0:
                kpis = self.calculate_kpis(timestamp)
                for key, value in kpis.items():
                    self.digital_twin.kpi_history[key].append(value)
        self.current_time = start_time + timedelta(seconds=duration_seconds)
        self.simulation_step += 1
        return {
            "step": self.simulation_step,
            "data_points_generated": data_points,
            "simulation_time": time.perf_counter() - step_start,
            "current_time": self.current_time,
        }

    def run_what_if_analysis(
        self, scenario_name: str, modifications: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Run a scenario on a temporary state and restore the baseline afterward."""
        twin = self.digital_twin
        original_state = {
            "vehicles": copy.deepcopy(twin.vehicles),
            "charging_stations": copy.deepcopy(twin.charging_stations),
            "environment": copy.deepcopy(twin.environment),
            "iot_data_stream": copy.deepcopy(twin.iot_data_stream),
            "kpi_history": copy.deepcopy(twin.kpi_history),
            "current_time": self.current_time,
            "simulation_step": self.simulation_step,
        }
        print(f"Running what-if analysis: {scenario_name}")
        try:
            if "add_charging_station" in modifications:
                new_station = copy.deepcopy(modifications["add_charging_station"])
                if any(station.station_id == new_station.station_id for station in twin.charging_stations):
                    raise ValueError("Charging-station identifiers must be unique")
                twin.charging_stations.append(new_station)
            if "change_energy_prices" in modifications:
                price_change = float(modifications["change_energy_prices"])
                if price_change < -1.0:
                    raise ValueError("The price change cannot reduce prices below zero")
                for station in twin.charging_stations:
                    station.energy_price *= 1.0 + price_change
            if "improve_battery_efficiency" in modifications:
                improvement = float(modifications["improve_battery_efficiency"])
                if not 0.0 <= improvement <= 1.0:
                    raise ValueError("Efficiency improvement must be between 0 and 1")
                for vehicle in twin.vehicles:
                    vehicle.energy_consumption_rate *= 1.0 - improvement

            simulation_results = self.run_simulation_step(duration_minutes=30)
            scenario_kpis = self.calculate_kpis(self.current_time)
            return {
                "scenario": scenario_name,
                "modifications": modifications,
                "results": simulation_results,
                "kpis": scenario_kpis,
            }
        finally:
            twin.vehicles = original_state["vehicles"]
            twin.charging_stations = original_state["charging_stations"]
            twin.environment = original_state["environment"]
            twin.iot_data_stream = original_state["iot_data_stream"]
            twin.kpi_history = original_state["kpi_history"]
            self.current_time = original_state["current_time"]
            self.simulation_step = original_state["simulation_step"]


orchestrator = DigitalTwinOrchestrator(digital_twin)
print("Created the digital-twin orchestrator.")

## 6. Run a simulation step

The duration argument is expressed in minutes and converted to one-second sensor intervals. Each interval generates readings, synchronizes recent values into the virtual assets, and records KPIs every ten simulated seconds. The simulation clock advances by the same elapsed duration as the generated timestamps.

Because this is a state-and-monitoring simulation rather than a vehicle-motion model, speed and battery state change only through generated sensor noise and synchronization. The KPI values should therefore be read as simulated indicators and estimates, not as evidence of actual deliveries, energy consumption, or service performance.

In [ ]:
print("Starting the digital-twin simulation")
print("=" * 48)
initial_results = orchestrator.run_simulation_step(duration_minutes=60)
print("\nInitial simulation results:")
print(f"Step: {initial_results['step']}")
print(f"Synthetic data points generated: {initial_results['data_points_generated']}")
print(f"Wall-clock runtime: {initial_results['simulation_time']:.3f} seconds")
print(f"Simulated time at completion: {initial_results['current_time']}")

current_kpis = orchestrator.calculate_kpis(orchestrator.current_time)
print("\nCurrent simulated indicators:")
print(f"Fleet energy deficit from full charge: {current_kpis['total_energy_consumed']:.2f} kWh")
print(f"Average battery level: {current_kpis['average_battery_level']:.1f}%")
print(f"Fleet utilization: {current_kpis['fleet_utilization']:.1%}")
print(f"Charging-station utilization: {current_kpis['charging_station_utilization']:.1%}")
print(f"Illustrative carbon estimate: {current_kpis['carbon_emissions']:.2f} kg CO2")
print(f"Estimated short-window charging cost: {current_kpis['operational_costs']:.2f} currency units")
print(f"Assumed service level: {current_kpis['service_level']:.1%}")

## 7. Examine isolated what-if scenarios and predictions

A what-if analysis temporarily applies a modification, runs the same simulation workflow, and reports the scenario's KPIs. The baseline fleet, histories, stream, clock, and step counter are restored after each scenario, including when a scenario raises an error. This isolation makes each scenario a separate comparison against the same starting state; random sensor readings can still differ because the random-number stream continues to advance.

The scenarios below explore an additional station, a price change, and a lower vehicle consumption rate. They do not constitute an optimization method or establish that a change improves real-world performance. The prediction outputs are heuristic rankings and estimates.

In [ ]:
new_station = ChargingStationTwin(
    station_id=3,
    location=(10.0, 15.0),
    capacity=2,
    available_points=2,
    charging_rate=150.0,
    queue_length=0,
    utilization_rate=0.0,
    energy_price=0.35,
    grid_carbon_intensity=140.0,
    sensors={"power_output": 148.5, "efficiency": 0.98, "availability": 1.0},
)

scenario1_results = orchestrator.run_what_if_analysis(
    "Add a high-speed charging station",
    {"add_charging_station": new_station},
)
print("\nScenario 1 indicators:")
print(f"Charging-station utilization: {scenario1_results['kpis']['charging_station_utilization']:.1%}")
print(f"Estimated short-window charging cost: {scenario1_results['kpis']['operational_costs']:.2f} currency units")

scenario2_results = orchestrator.run_what_if_analysis(
    "Increase energy prices by 20 percent",
    {"change_energy_prices": 0.20},
)
print("\nScenario 2 indicators:")
print(f"Estimated short-window charging cost: {scenario2_results['kpis']['operational_costs']:.2f} currency units")
print(f"Fleet utilization: {scenario2_results['kpis']['fleet_utilization']:.1%}")

scenario3_results = orchestrator.run_what_if_analysis(
    "Reduce vehicle consumption rates by 10 percent",
    {"improve_battery_efficiency": 0.10},
)
print("\nScenario 3 indicators:")
print(f"Fleet energy deficit from full charge: {scenario3_results['kpis']['total_energy_consumed']:.2f} kWh")
print(f"Illustrative carbon estimate: {scenario3_results['kpis']['carbon_emissions']:.2f} kg CO2")

print("\nRule-based predictive analytics:")
print("=" * 48)
charging_demand = predictive_analytics.predict_charging_demand(timedelta(hours=4))
print("Illustrative charging-demand index for the next four hours:")
for station_id, demand in charging_demand.items():
    print(f"  {station_id}: {demand:.1%}")

battery_degradation = predictive_analytics.predict_battery_degradation(
    digital_twin.vehicles[0], days_ahead=30
)
print("\nIllustrative battery-health projection for Vehicle 0:")
print(f"  Current health: {battery_degradation['current_health']:.1%}")
print(f"  Projected health after 30 days: {battery_degradation['predicted_health']:.1%}")
print(f"  Heuristic days to 80% health: {battery_degradation['days_to_80_percent']}")

optimal_charging = predictive_analytics.predict_optimal_charging_times(
    digital_twin.vehicles[0]
)
print("\nIllustrative charging-time ranking for Vehicle 0:")
print(f"  Lowest-scoring hour: {optimal_charging['best_hour']}:00")
print(f"  Highest-scoring hour: {optimal_charging['worst_hour']}:00")
print("  Three lowest-scoring hours:")
for rank, time_info in enumerate(optimal_charging["optimal_times"][:3], start=1):
    print(f"    {rank}. {time_info['hour']}:00 (score: {time_info['total_score']:.2f})")

## 8. Visualize the simulated system

The dashboard presents battery level and health, station utilization and energy price, recorded KPI trends, and a text summary of the current system. Battery-level bars use a percentage calculated from kWh level and capacity; battery-health values are also percentages. The station plot uses separate axes for utilization and price because the quantities have different units.

The displayed indicators summarize the simulator's current state. In particular, the KPI trends reflect the simulation's state-based proxies and should not be interpreted as measured fleet performance.

In [ ]:
def visualize_digital_twin(
    digital_twin: DigitalTwinSystem,
    orchestrator: DigitalTwinOrchestrator,
) -> None:
    """Plot fleet status, station status, KPI history, and system context."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Electric Vehicle Fleet — Integrated Digital-Twin Simulation",
        fontsize=16,
        fontweight="bold",
    )

    vehicle_ids = [f"V{vehicle.vehicle_id}" for vehicle in digital_twin.vehicles]
    battery_percentages = [
        100.0 * vehicle.battery_level / vehicle.battery_capacity
        for vehicle in digital_twin.vehicles
    ]
    health_percentages = [100.0 * vehicle.battery_health for vehicle in digital_twin.vehicles]
    positions = np.arange(len(vehicle_ids))
    width = 0.35
    ax1.bar(
        positions - width / 2,
        battery_percentages,
        width,
        label="Battery level (%)",
        alpha=0.75,
        color="tab:blue",
    )
    ax1_health = ax1.twinx()
    ax1_health.bar(
        positions + width / 2,
        health_percentages,
        width,
        label="Battery health (%)",
        alpha=0.75,
        color="tab:green",
    )
    ax1.set_xlabel("Vehicle")
    ax1.set_ylabel("Battery level (%)", color="tab:blue")
    ax1_health.set_ylabel("Battery health (%)", color="tab:green")
    ax1.set_title("Fleet battery status")
    ax1.set_xticks(positions)
    ax1.set_xticklabels(vehicle_ids)
    ax1.legend(loc="upper left")
    ax1_health.legend(loc="upper right")
    ax1.grid(True, alpha=0.3)

    station_ids = [f"CS{station.station_id}" for station in digital_twin.charging_stations]
    utilizations = [100.0 * station.utilization_rate for station in digital_twin.charging_stations]
    prices = [station.energy_price for station in digital_twin.charging_stations]
    bars = ax2.bar(station_ids, utilizations, alpha=0.75, label="Utilization (%)")
    ax2_price = ax2.twinx()
    ax2_price.plot(
        station_ids,
        prices,
        color="black",
        marker="o",
        linewidth=2,
        label="Energy price (currency units/kWh)",
    )
    ax2.set_xlabel("Charging station")
    ax2.set_ylabel("Utilization (%)")
    ax2_price.set_ylabel("Energy price (currency units/kWh)")
    ax2.set_title("Charging-station status")
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc="upper left")
    ax2_price.legend(loc="upper right")
    for bar, utilization in zip(bars, utilizations):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{utilization:.1f}%",
            ha="center",
            va="bottom",
        )

    history = digital_twin.kpi_history
    if history["average_battery_level"]:
        time_points = np.arange(len(history["average_battery_level"]))
        ax3.plot(
            time_points,
            history["average_battery_level"],
            label="Average battery level (%)",
            linewidth=2,
        )
        ax3.plot(
            time_points,
            [100.0 * value for value in history["fleet_utilization"]],
            label="Fleet utilization (%)",
            linewidth=2,
        )
        ax3.plot(
            time_points,
            [100.0 * value for value in history["charging_station_utilization"]],
            label="Charging-station utilization (%)",
            linewidth=2,
        )
        ax3.set_xlabel("Recorded KPI interval (10 simulated seconds)")
        ax3.set_ylabel("Percentage (%)")
        ax3.set_title("Recorded KPI trends")
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    else:
        ax3.text(
            0.5,
            0.5,
            "No KPI history is available.",
            ha="center",
            va="center",
            transform=ax3.transAxes,
        )
        ax3.set_title("Recorded KPI trends")

    current_kpis = orchestrator.calculate_kpis(orchestrator.current_time)
    metrics_text = (
        "Digital-twin simulation indicators\n"
        "──────────────────────────────────\n"
        f"Fleet energy deficit: {current_kpis['total_energy_consumed']:.2f} kWh\n"
        f"Average battery level: {current_kpis['average_battery_level']:.1f}%\n"
        f"Fleet utilization: {current_kpis['fleet_utilization']:.1%}\n"
        f"Station utilization: {current_kpis['charging_station_utilization']:.1%}\n"
        f"Illustrative carbon estimate: {current_kpis['carbon_emissions']:.2f} kg CO2\n"
        f"Estimated charging cost: {current_kpis['operational_costs']:.2f} currency units\n"
        f"Assumed service level: {current_kpis['service_level']:.1%}\n\n"
        "System configuration\n"
        "─────────────────────\n"
        f"Vehicles: {len(digital_twin.vehicles)}\n"
        f"Charging stations: {len(digital_twin.charging_stations)}\n"
        f"Nominal vehicle sensors: {len(digital_twin.vehicles) * 5}\n"
        f"Nominal station sensors: {len(digital_twin.charging_stations) * 3}\n"
        "Environmental sensors: 2\n"
        f"Synchronization interval: {digital_twin.synchronization_interval:.1f} s\n"
        f"Prediction horizon context: {digital_twin.prediction_horizon}\n"
        f"Retained data points: {len(digital_twin.iot_data_stream)}\n\n"
        "Simulation status\n"
        "─────────────────\n"
        f"Step: {orchestrator.simulation_step}\n"
        f"Simulated clock: {orchestrator.current_time.strftime('%H:%M:%S')}\n"
        "Synthetic data; estimates are illustrative."
    )
    ax4.text(
        0.04,
        0.96,
        metrics_text,
        transform=ax4.transAxes,
        fontsize=9,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightblue", "alpha": 0.8},
    )
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)
    ax4.axis("off")
    ax4.set_title("System summary")

    plt.tight_layout()
    plt.show()


visualize_digital_twin(digital_twin, orchestrator)

## 9. Interpret performance indicators

The analysis below summarizes the recent synthetic data stream, the age of the latest virtual-asset updates, and simple fleet-state indicators. A data-quality score describes the simulator's assigned quality values; it does not validate the sensor model. Synchronization accuracy is the share of assets updated within the stated time window. The battery prediction error compares a simple recent-history average with the latest recorded value, so it is a descriptive check rather than an independent forecast evaluation.

Recommendations are conditional rules applied to the simulated state. If no condition is met, the corresponding recommendation is simply not printed. This keeps the interpretation tied to the example's stated thresholds instead of implying a comprehensive operational decision system.

In [ ]:
def analyze_digital_twin_performance(
    digital_twin: DigitalTwinSystem,
    orchestrator: DigitalTwinOrchestrator,
) -> None:
    """Summarize synthetic data quality, synchronization, and fleet indicators."""
    print("Digital-twin simulation performance analysis:")
    print("=" * 54)

    recent_data = [
        point
        for point in digital_twin.iot_data_stream
        if 0.0 <= (orchestrator.current_time - point.timestamp).total_seconds() <= 600.0
    ]
    if recent_data:
        average_quality = float(np.mean([point.quality_score for point in recent_data]))
        data_rate = len(recent_data) / 600.0
        print("\nSynthetic data quality:")
        print(f"  Mean assigned quality score: {average_quality:.1%}")
        print(f"  Retained readings per 600-second window: {data_rate:.1f} per second")
        print(f"  Data points currently retained: {len(digital_twin.iot_data_stream)}")
    else:
        print("\nNo readings fall within the recent 600-second window.")

    vehicle_updates = [
        vehicle
        for vehicle in digital_twin.vehicles
        if 0.0 <= (orchestrator.current_time - vehicle.last_update).total_seconds() <= 5.0
    ]
    station_updates = [
        station
        for station in digital_twin.charging_stations
        if 0.0 <= (orchestrator.current_time - station.last_update).total_seconds() <= 5.0
    ]
    asset_count = len(digital_twin.vehicles) + len(digital_twin.charging_stations)
    synchronization_accuracy = (
        (len(vehicle_updates) + len(station_updates)) / asset_count
        if asset_count
        else 0.0
    )
    print("\nSynchronization status:")
    print(f"  Assets updated within five seconds: {synchronization_accuracy:.1%}")
    print(f"  Vehicles updated: {len(vehicle_updates)}/{len(digital_twin.vehicles)}")
    print(f"  Stations updated: {len(station_updates)}/{len(digital_twin.charging_stations)}")
    print(f"  Configured synchronization interval: {digital_twin.synchronization_interval:.1f} second(s)")

    battery_history = digital_twin.kpi_history["average_battery_level"]
    if len(battery_history) > 5:
        recent_battery = battery_history[-5:]
        predicted_value = float(np.mean(recent_battery[:-1]))
        actual_value = recent_battery[-1]
        prediction_error = (
            abs(predicted_value - actual_value) / abs(actual_value)
            if not np.isclose(actual_value, 0.0)
            else float("nan")
        )
        print("\nDescriptive recent-history check:")
        print(f"  Relative battery-level error: {prediction_error:.1%}")
        print(f"  Configured prediction-horizon context: {digital_twin.prediction_horizon}")

    energy_deficit = sum(
        max(0.0, vehicle.battery_capacity - vehicle.battery_level)
        for vehicle in digital_twin.vehicles
    )
    average_health = float(np.mean([
        vehicle.battery_health for vehicle in digital_twin.vehicles
    ]))
    average_consumption_rate = float(np.mean([
        vehicle.energy_consumption_rate for vehicle in digital_twin.vehicles
    ]))
    print("\nFleet-state indicators:")
    print(f"  Energy deficit from full charge: {energy_deficit:.2f} kWh")
    print(f"  Average battery health: {average_health:.1%}")
    print(f"  Mean base consumption rate: {average_consumption_rate:.3f} kWh/km")

    print("\nThreshold-based recommendations:")
    recommendations = []
    low_battery = [vehicle for vehicle in digital_twin.vehicles if vehicle.battery_level < 30.0]
    if low_battery:
        recommendations.append(f"{len(low_battery)} vehicle(s) have less than 30 kWh remaining.")
    highly_utilized = [
        station for station in digital_twin.charging_stations
        if station.utilization_rate > 0.8
    ]
    if highly_utilized:
        recommendations.append(f"{len(highly_utilized)} station(s) exceed 80% utilization.")
    degraded = [vehicle for vehicle in digital_twin.vehicles if vehicle.battery_health < 0.9]
    if degraded:
        recommendations.append(f"{len(degraded)} vehicle(s) have battery health below 90%.")
    if average_health < 0.95:
        recommendations.append("Consider reviewing the illustrative battery-health assumptions.")
    carbon_intensity = digital_twin.environment.carbon_intensity
    if carbon_intensity > 200:
        recommendations.append(
            f"Grid carbon intensity is high at {carbon_intensity:.0f} gCO2/kWh."
        )
    elif carbon_intensity < 100:
        recommendations.append(
            f"Grid carbon intensity is low at {carbon_intensity:.0f} gCO2/kWh."
        )
    if recommendations:
        for recommendation in recommendations:
            print(f"  - {recommendation}")
    else:
        print("  No configured threshold triggered a recommendation.")


analyze_digital_twin_performance(digital_twin, orchestrator)

## 10. Tier 5: capabilities, trade-offs, and appropriate use

An integrated digital twin extends fleet analysis from a single planning calculation to a persistent, data-oriented representation of vehicles, charging infrastructure, and operating conditions. In this notebook, the integration is demonstrated with synthetic data and a small simulation. A deployed twin would require reliable interfaces to physical assets, validated models, data governance, operational safeguards, and explicit treatment of uncertainty.

### Capabilities illustrated

- **Asset representation:** vehicle, station, and environmental attributes are organized in linked virtual models.
- **Sensor-data workflow:** timestamped synthetic readings are generated, retained, and used to update virtual states.
- **Energy estimates:** route consumption and charging calculations make units and assumptions explicit.
- **Predictive heuristics:** simple rules illustrate how current state and time patterns can inform forecasts.
- **Scenario analysis:** temporary changes can be assessed against a restored baseline.
- **KPI monitoring:** multiple indicators are recorded and displayed over simulated time.

### Trade-offs and limitations

| Aspect | Potential value | Important limitation in this example |
|---|---|---|
| Monitoring | A unified view of simulated assets | No physical sensors or real-time communications are connected |
| Prediction | Heuristics expose candidate future conditions | No fitted model, uncertainty estimate, or forecast validation is provided |
| Integration | Asset and environment state share one system representation | Real deployments require data standards, security, and failure handling |
| What-if analysis | Scenario inputs can be tested without changing the baseline | Scenario outputs are synthetic estimates, not causal evidence |
| Strategic planning | Persistent state can support operational questions | The example does not optimize routes, charging schedules, or infrastructure |

### Appropriate interpretation

This tier is most useful when an organization needs a connected view of fleet assets and has the data, infrastructure, and governance required to maintain it. The notebook demonstrates the architecture and reasoning pattern, not the performance of a production digital twin. More detailed physical models, additional constraints, validated sensor data, and larger fleet instances are natural extensions of the same digital-twin workflow; they are outside the implementation here.